# 面试问题：Agent 怎样把反馈写入长期记忆，同时处理冲突、权限、时效与投毒？

**一句话回答。** 长期记忆不是把每轮对话直接 append 到向量库。每条候选记忆要绑定 subject、属性、值、来源、证据、写入者、revision、有效期、权限和验证状态；写入前用 admission gate 过滤未确认或越权反馈，读取时按 ACL、时效和 revision 解析冲突。模型只能提出候选，不能单独裁决“最新事实”。

本 Notebook 只以受控小数据实现数据合同、状态机和断言，不调用大模型、真实 OAuth、真实文件或外部工具。断言验证机制不代表生产性能、安全或合规结论。

**资料入口。** [Memory conflict resolution 研究](https://arxiv.org/abs/2606.01435) 指出动态事实不能只交给 LLM 判断新旧；本例把来源、revision、验证与 ACL 放入确定性 memory resolver。

In [ ]:
question = "Agent 反馈记忆冲突与来源治理"  # 执行本行的状态、计算或校验逻辑。
assert "记忆" in question  # 执行本行的状态、计算或校验逻辑。
assert 10 - 4 == 6  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. 反馈不是事实

用户纠错、tool observation、人工审核和模型自省的可信度不同。把“用户说地址改了”立即广播给所有任务会制造投毒与错误个性化；系统应保留 raw feedback 的来源与 scope，经权威工具、人工或可验证规则确认后，再生成可被检索的 canonical memory。

In [ ]:
feedback = [{"id": "f-1", "subject": "customer-7", "predicate": "address", "value": "旧地址", "source": "tool", "verified": True, "revision": 1, "tenant": "t-1"}, {"id": "f-2", "subject": "customer-7", "predicate": "address", "value": "新地址", "source": "user", "verified": False, "revision": 2, "tenant": "t-1"}]  # 执行本行的状态、计算或校验逻辑。
assert len(feedback) == 2  # 执行本行的状态、计算或校验逻辑。
assert feedback[0]["verified"]  # 执行本行的状态、计算或校验逻辑。
assert feedback[1]["source"] == "user"  # 执行本行的状态、计算或校验逻辑。

## 2. 写入合同

memory key 应是明确 subject–predicate，而不是一段模糊自然语言。记录 revision 和 evidence id 使同一事实可被更新、撤销和审计；记录 tenant/role/TTL 防止跨用户泄露和历史信息污染。没有可解释 key 的摘要应只作为短期提示，不应升级为持久事实。

In [ ]:
def memory_key(item):  # 执行本行的状态、计算或校验逻辑。
    return (item["tenant"], item["subject"], item["predicate"])  # 执行本行的状态、计算或校验逻辑。
assert memory_key(feedback[0]) == ("t-1", "customer-7", "address")  # 执行本行的状态、计算或校验逻辑。
assert memory_key(feedback[0]) == memory_key(feedback[1])  # 执行本行的状态、计算或校验逻辑。
assert feedback[1]["revision"] > feedback[0]["revision"]  # 执行本行的状态、计算或校验逻辑。

## 3. 冲突解析

对同一 key 的多个值，先过滤 tombstone、过期、未验证和无权限候选，再按可信来源与单调 revision 选择当前版本；不可比较的冲突要返回澄清或人工队列。不要仅按 embedding 相似度或 LLM 语言流畅度决定新旧，因为两者都不能证明版本语义。

In [ ]:
def admit(item):  # 执行本行的状态、计算或校验逻辑。
    return item["verified"] and item["source"] in {"tool", "human"} and item["revision"] > 0  # 执行本行的状态、计算或校验逻辑。
admitted = [item for item in feedback if admit(item)]  # 执行本行的状态、计算或校验逻辑。
assert [item["id"] for item in admitted] == ["f-1"]  # 执行本行的状态、计算或校验逻辑。
assert not admit(feedback[1])  # 执行本行的状态、计算或校验逻辑。
assert admit(feedback[0])  # 执行本行的状态、计算或校验逻辑。

## 4. 读取与最小暴露

检索不仅按相似度，还必须在召回前做 tenant、role、scope 和时间过滤。即使 Agent 有一个任务需要客户地址，也不意味着另一个用户或无关 tool 能看到它；最终答案最好附上 memory id/revision，让后续 action 与审计能回到同一事实版本。

In [ ]:
confirmed = {**feedback[1], "id": "f-3", "source": "human", "verified": True}  # 执行本行的状态、计算或校验逻辑。
records = admitted + [confirmed]  # 执行本行的状态、计算或校验逻辑。
def resolve(items):  # 执行本行的状态、计算或校验逻辑。
    return max(items, key=lambda item: item["revision"])  # 执行本行的状态、计算或校验逻辑。
assert resolve(records)["id"] == "f-3"  # 执行本行的状态、计算或校验逻辑。
assert resolve(records)["value"] == "新地址"  # 执行本行的状态、计算或校验逻辑。
assert len(records) == 2  # 执行本行的状态、计算或校验逻辑。

## 5. 更正与撤销

更正应追加新 revision 并 tombstone 旧版本，而不是原地无痕覆盖。这样可以复放旧决策、解释为什么过去答案不同，并让缓存/索引按 revision 失效。删除请求还需要从派生摘要、索引和训练/评测管线继续追踪，不能只删一条向量。

In [ ]:
def visible(item, tenant, role, now):  # 执行本行的状态、计算或校验逻辑。
    return item["tenant"] == tenant and role == "support" and item.get("expires", 999) > now  # 执行本行的状态、计算或校验逻辑。
current = {**resolve(records), "expires": 50}  # 执行本行的状态、计算或校验逻辑。
assert visible(current, "t-1", "support", 10)  # 执行本行的状态、计算或校验逻辑。
assert not visible(current, "t-2", "support", 10)  # 执行本行的状态、计算或校验逻辑。
assert not visible(current, "t-1", "guest", 10)  # 执行本行的状态、计算或校验逻辑。

## 6. 投毒与评测

评测要专门加入未经验证的用户主张、过期地址、跨租户记录和相同 key 的冲突 revision，报告拒绝率、正确解析率、错误暴露率和回滚时间。性能变好不能掩盖 provenance 丢失：更高 recall 若跨越权限边界，仍是失败。

In [ ]:
tombstones = {"f-1"}  # 执行本行的状态、计算或校验逻辑。
def active(items, removed):  # 执行本行的状态、计算或校验逻辑。
    return [item for item in items if item["id"] not in removed]  # 执行本行的状态、计算或校验逻辑。
active_records = active(records, tombstones)  # 执行本行的状态、计算或校验逻辑。
assert [item["id"] for item in active_records] == ["f-3"]  # 执行本行的状态、计算或校验逻辑。
assert "f-1" in tombstones  # 执行本行的状态、计算或校验逻辑。
assert resolve(active_records)["revision"] == 2  # 执行本行的状态、计算或校验逻辑。

## 7. 验收与边界

本例把 verification 和 revision 简化为小字典；真实系统还要签名来源、加密、访问审计、事件时间、并发写冲突、ANN 过滤正确性和法务删除流程。确定性 resolver 不能替代事实核验，但能避免把本可由数据合同处理的问题推给模型猜测。

In [ ]:
answer_evidence = {"memory_id": current["id"], "revision": current["revision"], "tenant": current["tenant"]}  # 执行本行的状态、计算或校验逻辑。
assert answer_evidence["memory_id"] == "f-3"  # 执行本行的状态、计算或校验逻辑。
assert answer_evidence["revision"] == 2  # 执行本行的状态、计算或校验逻辑。
assert answer_evidence["tenant"] == "t-1"  # 执行本行的状态、计算或校验逻辑。

## 8. 面试追问

回答时还应区分教学状态机与生产系统：前者用小数据证明拒绝条件和版本绑定，后者还要覆盖并发、网络故障、机密管理、审计留存与真实依赖的集成测试。任何无法由当前证据确认的状态，都应显式返回未验证、降级或人工升级，而不是由模型补全。

In [ ]:
evaluation = {"verified_current": resolve(active_records)["value"] == "新地址", "poison_rejected": not admit(feedback[1]), "cross_tenant_blocked": not visible(current, "t-2", "support", 10)}  # 执行本行的状态、计算或校验逻辑。
assert all(evaluation.values())  # 执行本行的状态、计算或校验逻辑。
assert evaluation["poison_rejected"]  # 执行本行的状态、计算或校验逻辑。
assert evaluation["cross_tenant_blocked"]  # 执行本行的状态、计算或校验逻辑。

## 面试总结

高质量回答应先给出模型或 Agent 的责任边界，再说明数据合同、状态转换、确定性 verifier 和失败处理，最后明确性能、权限与现实系统依赖的验证方法。不要把一次函数返回、模型文本或受控小样本断言误称为线上正确性。